# 06.1 · EDA — Silver physical_itens_venda_caixa
**Projeto:** Estágio Engenharia de Dados · **Squad:** 3 — Batch · **Autor:** Alexandro Silva

Análise exploratória da camada Silver, antes da etapa de Machine Learning. Três objetivos, conforme
orientação do professor:
1. **Adequação dos dados**: confirmar que a Silver está íntegra (nulos, chave, flags, cobertura do catálogo, partições).
2. **Conhecer a tabela**: distribuições, unidades de medida, comportamento no tempo, produtos e categorias.
3. **Insights para o modelo**: problema **P2 · Segmentação de cestas de compra** (planilha de ML da Squad 3),
   modelo **B: Gaussian Mixture (GMM)**. As features por transação são a % da receita por `categoria_pai`,
   o nº de itens e o `valor_total`.

Este notebook **não grava nada**: só lê e analisa. Nenhuma linha é removida; as flags da Silver são
quantificadas e a decisão de excluí-las fica para a etapa de ML.

## Perguntas que a EDA responde
| Nº | Pergunta | Etapa |
|---|---|---|
| 1 | A Silver tem nulos, chave duplicada ou partições vazias? | 3 |
| 2 | Quanto pesam as flags `flag_valor_inconsistente` e `flag_quantidade_fracionada_un`? | 3 |
| 3 | Como se distribuem quantidade, preço e valor do item? Há outliers? | 4 |
| 4 | Há sazonalidade em volume, receita e ticket? | 5 |
| 5 | Todo SKU tem `categoria_pai`? Qual a concentração de receita por categoria e por SKU? | 6 |
| 6 | Como são as cestas (nº de itens, valor, mix de categorias)? | 7 |
| 7 | As features do GMM precisam de transformação (assimetria, esparsidade, colinearidade)? | 8 |

## Origem
| Item | Definição |
|---|---|
| Silver de itens | `abfss://squad3@internshipdatalake.dfs.core.windows.net/silver/physical_itens_venda_caixa` |
| Catálogo (categoria) | **provisório:** `workspace.squad3.ecommerce_produtos` + `ecommerce_categorias` + `physical_produtos_pereciveis` |
| Vendas (cabeçalho) | **provisório:** `workspace.squad3.physical_vendas_caixa` (só na conferência opcional da etapa 7) |

## Etapas
1. Credenciais e caminhos
2. Leitura da Silver e do catálogo
3. Qualidade e adequação
4. Perfil das colunas numéricas
5. Comportamento no tempo
6. Produtos e categorias
7. Visão por cesta (features do GMM)
8. Implicações para o modelo
9. Checklist de adequação e conclusões

## 1. Credenciais e caminhos
Mesmo padrão do notebook 06. As tabelas de referência continuam **provisórias** (cópias derivadas da RAW)
até as duplas responsáveis publicarem suas Silver.

In [0]:
# Célula 1 — Credenciais e caminhos
from dotenv import load_dotenv
import os

load_dotenv("/Workspace/Users/alexandrofs31@gmail.com/.env")

CLIENT_ID       = os.getenv("ADLS_CLIENT_ID")
TENANT_ID       = os.getenv("ADLS_TENANT_ID")
CLIENT_SECRET   = os.getenv("ADLS_CLIENT_SECRET")
STORAGE_ACCOUNT = os.getenv("ADLS_STORAGE_ACCOUNT", "internshipdatalake")

faltando = [n for n, v in {"ADLS_CLIENT_ID": CLIENT_ID, "ADLS_TENANT_ID": TENANT_ID,
                           "ADLS_CLIENT_SECRET": CLIENT_SECRET}.items() if not v]
if faltando:
    raise ValueError(f"Variáveis ausentes no .env: {faltando}")

TABELA          = "physical_itens_venda_caixa"
CONTAINER_SQUAD = "squad3"
BASE_SQUAD      = f"abfss://{CONTAINER_SQUAD}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
CAMINHO_SILVER  = f"{BASE_SQUAD}/silver/{TABELA}"

# PROVISÓRIO: trocar pela Silver das duplas responsáveis quando forem publicadas
TABELA_VENDAS_REF     = "workspace.squad3.physical_vendas_caixa"
TABELA_PERECIVEIS_REF = "workspace.squad3.physical_produtos_pereciveis"
TABELA_PRODUTOS_REF   = "workspace.squad3.ecommerce_produtos"
TABELA_CATEGORIAS_REF = "workspace.squad3.ecommerce_categorias"

# Tamanho máximo das amostras levadas ao pandas para gráficos e correlação
AMOSTRA_MAX = 200_000
SEED = 42

sufixo = f"{STORAGE_ACCOUNT}.dfs.core.windows.net"
opcoes_adls = {
    f"fs.azure.account.auth.type.{sufixo}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{sufixo}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{sufixo}": CLIENT_ID,
    f"fs.azure.account.oauth2.client.secret.{sufixo}": CLIENT_SECRET,
    f"fs.azure.account.oauth2.client.endpoint.{sufixo}": f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token",
}

print(f"📦 Silver   : {CAMINHO_SILVER}")
print(f"📚 Catálogo : {TABELA_PRODUTOS_REF} + {TABELA_PERECIVEIS_REF}")

## 2. Leitura da Silver e do catálogo
Lê a Silver de itens e monta o catálogo unificado (secos + perecíveis) com `sku` e `categoria_pai`:
- **e-commerce:** `ecommerce_produtos.id_categoria` aponta para uma subcategoria; a categoria pai vem de
  `ecommerce_categorias.nome_categoria_pai` (ou do próprio `nome_categoria`, quando o produto já está numa categoria de topo).
- **perecíveis:** `physical_produtos_pereciveis` já traz `categoria_pai`.

A célula mostra as categorias de cada origem lado a lado, para conferir se seguem o mesmo padrão de nomes.

SKUs presentes **nos dois catálogos com categorias diferentes** são contados, porque o `dropDuplicates`
escolheria uma delas ao acaso.

In [0]:
# Célula 2 — Leitura da Silver e do catálogo unificado
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.max_columns", 50)

df = spark.read.format("delta").options(**opcoes_adls).load(CAMINHO_SILVER)
total_itens = df.count()
print(f"✅ Silver lida: {total_itens:,} linhas | {len(df.columns)} colunas")
df.printSchema()

# e-commerce: produto -> id_categoria (subcategoria) -> categoria pai em ecommerce_categorias.
# Se o id_categoria do produto já for uma categoria de topo (sem pai), ela mesma é a categoria_pai.
df_categorias = spark.table(TABELA_CATEGORIAS_REF).select(
    F.col("id_categoria").cast("int").alias("id_categoria"),
    F.col("id_categoria_pai").cast("int").alias("id_categoria_pai"),
    F.coalesce(F.col("nome_categoria_pai"), F.col("nome_categoria")).alias("nome_pai"))

# A regra acima supõe 2 níveis. Se algum pai também tiver pai, a hierarquia é mais funda: avisa.
niveis_extras = (df_categorias.alias("f")
    .join(df_categorias.alias("p"), F.col("f.id_categoria_pai") == F.col("p.id_categoria"))
    .filter(F.col("p.id_categoria_pai").isNotNull()).count())
print(f"   Categorias com mais de 2 níveis : {niveis_extras} {'✅' if niveis_extras == 0 else '⚠️ revisar a regra'}")

cat_ecommerce = (spark.table(TABELA_PRODUTOS_REF)
    .withColumn("id_categoria", F.col("id_categoria").cast("int"))
    .join(df_categorias, on="id_categoria", how="left")
    .select(F.lower(F.trim(F.col("sku"))).alias("codigo_barras_produto"),
            F.lower(F.trim(F.col("nome_pai"))).alias("categoria_pai"),
            F.lit("ecommerce_produtos").alias("origem_catalogo")))

# perecíveis: já trazem categoria_pai
cat_pereciveis = (spark.table(TABELA_PERECIVEIS_REF)
    .select(F.lower(F.trim(F.col("sku"))).alias("codigo_barras_produto"),
            F.lower(F.trim(F.col("categoria_pai"))).alias("categoria_pai"),
            F.lit("physical_produtos_pereciveis").alias("origem_catalogo")))

df_cat_bruto = cat_ecommerce.unionByName(cat_pereciveis)

print("\n📚 categoria_pai por origem (confira se os nomes seguem o mesmo padrão nas duas):")
display(df_cat_bruto.groupBy("origem_catalogo", "categoria_pai").count()
        .orderBy("origem_catalogo", "categoria_pai").toPandas())

sem_pai = df_cat_bruto.filter(F.col("categoria_pai").isNull()).count()
print(f"   SKUs do catálogo sem categoria_pai: {sem_pai:,}")

conflitos = (df_cat_bruto.groupBy("codigo_barras_produto")
             .agg(F.countDistinct("categoria_pai").alias("n_cat"))
             .filter("n_cat > 1").count())

df_catalogo = (df_cat_bruto
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("codigo_barras_produto").orderBy("origem_catalogo")))
    .filter("_rn = 1").drop("_rn"))

print(f"   SKUs no catálogo unificado        : {df_catalogo.count():,}")
print(f"   SKUs com categoria conflitante    : {conflitos:,}")
print(f"   categoria_pai distintas           : {df_catalogo.select('categoria_pai').distinct().count():,}")

## 3. Qualidade e adequação
Reconfere na Silver gravada o que a Silver promete: chave única, ausência de nulos nas colunas
obrigatórias, valores positivos. Também mede o peso das flags **em linhas e em receita**, porque uma
flag rara em linhas pode concentrar receita, e vice-versa.

In [0]:
# Célula 3a — Nulos, chave e domínio
nulos = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0].asDict()
tab_nulos = (pd.DataFrame({"nulos": nulos})
             .assign(pct=lambda d: d["nulos"] / total_itens * 100)
             .sort_values("nulos", ascending=False))
print("🔎 Nulos por coluna:")
display(tab_nulos)

chaves = df.select("id_item_venda").distinct().count()
dominio = df.agg(
    F.sum((F.col("quantidade") <= 0).cast("int")).alias("quantidade_nao_positiva"),
    F.sum((F.col("preco_unitario_registro") <= 0).cast("int")).alias("preco_nao_positivo"),
    F.sum((F.col("valor_total_item") <= 0).cast("int")).alias("valor_nao_positivo"),
    F.countDistinct("id_transacao").alias("transacoes"),
    F.countDistinct("codigo_barras_produto").alias("skus_vendidos"),
).collect()[0].asDict()

print(f"\n🔑 id_item_venda distintos : {chaves:,} de {total_itens:,} linhas "
      f"{'✅' if chaves == total_itens else '⚠️ DUPLICADO'}")
for k, v in dominio.items():
    print(f"   {k:<26}: {v:,}")

In [0]:
# Célula 3b — Peso das flags e da ausência de unidade_medida (em linhas e em receita)
receita_total = float(df.agg(F.sum("valor_total_item")).collect()[0][0])

def peso(cond, nome):
    r = df.filter(cond).agg(F.count("*").alias("linhas"),
                            F.sum("valor_total_item").alias("receita"),
                            F.countDistinct("id_transacao").alias("transacoes")).collect()[0]
    receita = float(r["receita"] or 0)
    return {"situação": nome, "linhas": r["linhas"], "% linhas": r["linhas"] / total_itens * 100,
            "receita": receita, "% receita": receita / receita_total * 100,
            "transações afetadas": r["transacoes"],
            "% transações": r["transacoes"] / dominio["transacoes"] * 100}

tab_flags = pd.DataFrame([
    peso(F.col("flag_valor_inconsistente"), "flag_valor_inconsistente"),
    peso(F.col("flag_quantidade_fracionada_un"), "flag_quantidade_fracionada_un"),
    peso(F.col("unidade_medida").isNull(), "produto sem unidade_medida"),
])
print(f"💰 Receita total da Silver: R$ {receita_total:,.2f}\n")
display(tab_flags)

# Tamanho da divergência nas linhas com flag_valor_inconsistente
diverg = (df.filter("flag_valor_inconsistente")
    .withColumn("diferenca", F.abs(F.col("quantidade") * F.col("preco_unitario_registro")
                                   - F.col("valor_total_item")).cast("double"))
    .select("diferenca"))
if diverg.limit(1).count():
    print("\n📏 Divergência (R$) nas linhas com flag_valor_inconsistente:")
    display(diverg.summary("count", "mean", "min", "50%", "90%", "99%", "max").toPandas())

In [0]:
# Célula 3c — Partições ano / mes (cobertura e buracos na série)
df_part = (df.groupBy("ano", "mes")
    .agg(F.count("*").alias("itens"),
         F.countDistinct("id_transacao").alias("transacoes"),
         F.sum("valor_total_item").cast("double").alias("receita"))
    .orderBy("ano", "mes").toPandas())
df_part["periodo"] = pd.to_datetime(dict(year=df_part.ano, month=df_part.mes, day=1))

esperado = pd.date_range(df_part.periodo.min(), df_part.periodo.max(), freq="MS")
faltantes = sorted(set(esperado) - set(df_part.periodo))
print(f"📅 Período: {df_part.periodo.min():%m/%Y} a {df_part.periodo.max():%m/%Y} "
      f"| {len(df_part)} meses com dados | meses faltando: "
      f"{[f'{m:%m/%Y}' for m in faltantes] or 'nenhum ✅'}")
display(df_part)

## 4. Perfil das colunas numéricas
Estatísticas de `quantidade`, `preco_unitario_registro` e `valor_total_item`, com percentis e
**assimetria** (skewness). Valores de assimetria acima de 1 indicam cauda longa à direita, comum em
dados de varejo, e pedem transformação logarítmica antes de modelos que supõem distribuição normal, como o GMM.
Outliers são contados pelo critério de IQR (acima de Q3 + 1,5 × IQR), **só para reportar**.

In [0]:
# Célula 4a — Estatísticas descritivas, assimetria e outliers
NUM = ["quantidade", "preco_unitario_registro", "valor_total_item"]
df_num = df.select([F.col(c).cast("double").alias(c) for c in NUM])

display(df_num.summary("count", "mean", "stddev", "min", "1%", "25%", "50%", "75%", "99%", "max").toPandas())

linhas = []
for c in NUM:
    q1, q3 = df_num.approxQuantile(c, [0.25, 0.75], 0.001)
    lim = q3 + 1.5 * (q3 - q1)
    r = df_num.agg(F.skewness(c).alias("skew"), F.kurtosis(c).alias("kurt"),
                   F.sum((F.col(c) > lim).cast("int")).alias("out")).collect()[0]
    linhas.append({"coluna": c, "assimetria": r["skew"], "curtose": r["kurt"],
                   "limite IQR": lim, "outliers": r["out"], "% outliers": r["out"] / total_itens * 100})
display(pd.DataFrame(linhas))

In [0]:
# Célula 4b — Unidade de medida: quantidade inteira x fracionada
display(df.groupBy("unidade_medida")
    .agg(F.count("*").alias("linhas"),
         F.sum("valor_total_item").cast("double").alias("receita"),
         F.avg(F.col("quantidade").cast("double")).alias("qtd_media"),
         F.expr("percentile_approx(cast(quantidade as double), 0.5)").alias("qtd_mediana"),
         F.avg((F.col("quantidade") != F.floor("quantidade")).cast("int")).alias("pct_fracionada"))
    .withColumn("pct_linhas", F.col("linhas") / total_itens * 100)
    .orderBy(F.desc("linhas")).toPandas())

In [0]:
# Célula 4c — Histogramas (amostra), escala linear e logarítmica
frac = min(1.0, AMOSTRA_MAX / total_itens)
amostra_itens = df_num.sample(False, frac, seed=SEED).toPandas()
print(f"Amostra: {len(amostra_itens):,} linhas ({frac:.1%} da Silver)")

fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for i, c in enumerate(NUM):
    s = amostra_itens[c].dropna()
    axes[0, i].hist(s.clip(upper=s.quantile(0.99)), bins=60, color="#4C72B0")
    axes[0, i].set_title(f"{c} (até p99)")
    axes[1, i].hist(np.log1p(s), bins=60, color="#DD8452")
    axes[1, i].set_title(f"log1p({c})")
plt.tight_layout(); plt.show()

## 5. Comportamento no tempo
Volume de itens, transações, receita e ticket médio por mês. Interessa ao modelo porque uma segmentação
estável não deveria depender do mês: se o mix de categorias muda muito com a sazonalidade (festas de
fim de ano, por exemplo), os grupos podem refletir **época** em vez de **missão de compra**. Isso é
retomado na etapa 7.

In [0]:
# Célula 5 — Séries mensais
df_part["ticket_medio"] = df_part.receita / df_part.transacoes
df_part["itens_por_transacao"] = df_part.itens / df_part.transacoes

fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharex=True)
for ax, (c, t) in zip(axes.flat, [("transacoes", "Transações por mês"), ("receita", "Receita por mês (R$)"),
                                   ("ticket_medio", "Ticket médio (R$)"), ("itens_por_transacao", "Itens por transação")]):
    ax.plot(df_part.periodo, df_part[c], marker="o", color="#4C72B0"); ax.set_title(t); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

# Perfil por mês do ano (média entre anos): mostra sazonalidade
saz = df_part.assign(mes_ano=df_part.periodo.dt.month).groupby("mes_ano")[["transacoes", "ticket_medio", "itens_por_transacao"]].mean()
display((saz / saz.mean()).rename(columns=lambda c: f"{c} (índice, média=1)"))

## 6. Produtos e categorias
Cruza o SKU com o catálogo para obter a `categoria_pai`. A **cobertura** é decisiva para o modelo: item
sem categoria vira a categoria `sem_categoria` na etapa 7 e distorce o mix da cesta.
Também mede a concentração de receita (curva de Pareto) por SKU.

In [0]:
# Célula 6a — Cobertura do catálogo e receita por categoria_pai
df_enr = (df.join(F.broadcast(df_catalogo.select("codigo_barras_produto", "categoria_pai")),
                  on="codigo_barras_produto", how="left")
            .withColumn("categoria_pai", F.coalesce("categoria_pai", F.lit("sem_categoria"))))

df_categ = (df_enr.groupBy("categoria_pai")
    .agg(F.count("*").alias("itens"),
         F.countDistinct("codigo_barras_produto").alias("skus"),
         F.countDistinct("id_transacao").alias("transacoes"),
         F.sum("valor_total_item").cast("double").alias("receita"))
    .toPandas().sort_values("receita", ascending=False))
df_categ["% receita"] = df_categ.receita / df_categ.receita.sum() * 100
df_categ["% transações com a categoria"] = df_categ.transacoes / dominio["transacoes"] * 100

sem = df_categ.loc[df_categ.categoria_pai == "sem_categoria"]
print(f"🏷️ categoria_pai distintas nas vendas: {len(df_categ) - len(sem)}")
print(f"   Itens sem categoria: {int(sem.itens.sum()):,} "
      f"({sem.itens.sum() / total_itens * 100:.2f}% das linhas, {sem['% receita'].sum():.2f}% da receita)")
display(df_categ)

fig, ax = plt.subplots(figsize=(12, max(4, len(df_categ) * 0.35)))
ax.barh(df_categ.categoria_pai[::-1], df_categ["% receita"][::-1], color="#4C72B0")
ax.set_xlabel("% da receita"); ax.set_title("Receita por categoria_pai"); plt.tight_layout(); plt.show()

In [0]:
# Célula 6b — Concentração de receita por SKU (Pareto) e top 15
df_sku = (df_enr.groupBy("codigo_barras_produto", "categoria_pai")
    .agg(F.sum("valor_total_item").cast("double").alias("receita"), F.count("*").alias("itens"))
    .toPandas().sort_values("receita", ascending=False).reset_index(drop=True))
df_sku["% acumulado"] = df_sku.receita.cumsum() / df_sku.receita.sum() * 100

n80 = int((df_sku["% acumulado"] < 80).sum()) + 1
print(f"📦 SKUs vendidos: {len(df_sku):,} | {n80:,} SKUs ({n80 / len(df_sku):.1%}) fazem 80% da receita")
display(df_sku.head(15))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(np.arange(1, len(df_sku) + 1) / len(df_sku) * 100, df_sku["% acumulado"], color="#4C72B0")
ax.axhline(80, ls="--", c="grey"); ax.set_xlabel("% dos SKUs"); ax.set_ylabel("% acumulado da receita")
ax.set_title("Curva de Pareto — receita por SKU"); ax.grid(alpha=.3); plt.show()

## 7. Visão por cesta (features do GMM)
Agrega os itens por `id_transacao` e monta as features definidas na planilha de ML (as mesmas da dupla 2,
que usa K-Means, para que os resultados sejam comparáveis):

| Feature | Definição |
|---|---|
| `n_itens` | nº de linhas de item na transação |
| `valor_total` | soma de `valor_total_item` |
| `pct_<categoria>` | receita da categoria ÷ `valor_total` da transação |

Colunas extras **só para análise** (não entram no modelo sem decisão da dupla): `n_skus`, `n_categorias`,
`tem_flag_valor`, `tem_flag_fracionada`.

> Atenção: a Silver tem só os itens que passaram nas regras. Se um item da transação foi removido
> (por exemplo, preço ≤ 0), o `valor_total` da cesta fica menor que o do cabeçalho em `physical_vendas_caixa`.
> A célula 7c confere isso.

In [0]:
# Célula 7a — Construção da tabela de cestas
categorias = sorted(df_categ.categoria_pai.tolist())

base_cesta = (df_enr.groupBy("id_transacao")
    .agg(F.min("ano").alias("ano"), F.min("mes").alias("mes"),
         F.count("*").alias("n_itens"),
         F.countDistinct("codigo_barras_produto").alias("n_skus"),
         F.countDistinct("categoria_pai").alias("n_categorias"),
         F.sum("valor_total_item").cast("double").alias("valor_total"),
         F.max(F.col("flag_valor_inconsistente").cast("int")).alias("tem_flag_valor"),
         F.max(F.col("flag_quantidade_fracionada_un").cast("int")).alias("tem_flag_fracionada")))

receita_cat = (df_enr.groupBy("id_transacao").pivot("categoria_pai", categorias)
               .agg(F.sum(F.col("valor_total_item").cast("double"))).fillna(0.0))

def nome_pct(c):
    return "pct_" + "".join(ch if ch.isalnum() else "_" for ch in c)

cesta = base_cesta.join(receita_cat, on="id_transacao", how="inner")
for c in categorias:
    cesta = cesta.withColumn(nome_pct(c), F.when(F.col("valor_total") > 0, F.col(f"`{c}`") / F.col("valor_total")).otherwise(0.0))
cesta = cesta.drop(*categorias)
COLS_PCT = [nome_pct(c) for c in categorias]

total_cestas = cesta.count()
print(f"🧺 Cestas (transações): {total_cestas:,} | features de mix: {len(COLS_PCT)}")

In [0]:
# Célula 7b — Perfil das cestas
FEAT_BASE = ["n_itens", "n_skus", "n_categorias", "valor_total"]
display(cesta.select(FEAT_BASE).summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "90%", "99%", "max").toPandas())

perfil = cesta.agg(
    F.avg((F.col("n_itens") == 1).cast("int")).alias("pct_cestas_1_item"),
    F.avg((F.col("n_categorias") == 1).cast("int")).alias("pct_cestas_mono_categoria"),
    F.avg("tem_flag_valor").alias("pct_cestas_com_flag_valor"),
    F.avg("tem_flag_fracionada").alias("pct_cestas_com_flag_fracionada"),
    *[F.skewness(c).alias(f"skew_{c}") for c in ["n_itens", "valor_total"]],
    *[F.skewness(F.log1p(c)).alias(f"skew_log1p_{c}") for c in ["n_itens", "valor_total"]],
).collect()[0].asDict()
for k, v in perfil.items():
    print(f"   {k:<32}: {v * 100:6.2f}%" if k.startswith("pct") else f"   {k:<32}: {v:8.3f}")

# Distribuição de n_categorias por cesta
display(cesta.groupBy("n_categorias").count()
        .withColumn("pct", F.col("count") / total_cestas * 100).orderBy("n_categorias").toPandas())

In [0]:
# Célula 7c — Conferência opcional: valor da cesta (Silver) x valor da venda (cabeçalho)
vendas = spark.table(TABELA_VENDAS_REF)
col_valor = next((c for c in ["valor_total", "valor_total_venda", "valor_venda"] if c in vendas.columns), None)

if col_valor is None:
    print(f"ℹ️ {TABELA_VENDAS_REF} sem coluna de valor reconhecida; conferência pulada. Colunas: {vendas.columns}")
else:
    comp = (cesta.select("id_transacao", "valor_total")
        .join(vendas.select(F.trim("id_transacao").alias("id_transacao"),
                            F.col(col_valor).cast("double").alias("valor_cabecalho")),
              on="id_transacao", how="left")
        .withColumn("diferenca", F.col("valor_cabecalho") - F.col("valor_total")))
    r = comp.agg(
        F.sum(F.col("valor_cabecalho").isNull().cast("int")).alias("sem_cabecalho"),
        F.sum((F.abs("diferenca") > 0.05).cast("int")).alias("divergentes_acima_5_centavos"),
        F.sum("diferenca").alias("diferenca_total_R$")).collect()[0].asDict()
    print(f"🧾 Conferência com {TABELA_VENDAS_REF}.{col_valor}:")
    for k, v in r.items():
        print(f"   {k:<30}: {v:,.2f}" if isinstance(v, float) else f"   {k:<30}: {v:,}")

## 8. Implicações para o modelo (GMM)
O GMM assume que cada grupo segue uma distribuição normal multivariada. Três pontos da EDA afetam
diretamente essa suposição:
1. **Assimetria** de `n_itens` e `valor_total`: comparar a assimetria antes e depois de `log1p` (célula 7b).
2. **Esparsidade** das `pct_*`: categorias que aparecem em poucas cestas geram colunas quase todas zero,
   que não têm forma normal e podem virar componentes "artificiais". Candidatas a agrupar em `outras`.
3. **Colinearidade** das `pct_*`: as porcentagens somam 1 em toda cesta, então uma coluna é combinação
   exata das outras. Com covariância `full`, isso torna a matriz singular. Opções a decidir na etapa de
   ML: retirar uma categoria, aplicar transformação log-razão (CLR) ou reduzir com PCA.

In [0]:
# Célula 8a — Esparsidade das features de mix
esp = cesta.agg(*[F.avg((F.col(c) > 0).cast("int")).alias(c) for c in COLS_PCT],
                ).collect()[0].asDict()
media_quando_presente = cesta.agg(*[F.avg(F.when(F.col(c) > 0, F.col(c))).alias(c) for c in COLS_PCT]).collect()[0].asDict()
tab_esp = (pd.DataFrame({"% cestas com a categoria": {k: v * 100 for k, v in esp.items()},
                         "participação média quando presente": media_quando_presente})
           .sort_values("% cestas com a categoria", ascending=False))
tab_esp["rara (<5% das cestas)"] = tab_esp["% cestas com a categoria"] < 5
display(tab_esp)

In [0]:
# Célula 8b — Correlação entre features (amostra de cestas)
frac_c = min(1.0, AMOSTRA_MAX / total_cestas)
amostra_cesta = cesta.select(["n_itens", "valor_total"] + COLS_PCT).sample(False, frac_c, seed=SEED).toPandas()
amostra_cesta["log_n_itens"] = np.log1p(amostra_cesta.pop("n_itens"))
amostra_cesta["log_valor_total"] = np.log1p(amostra_cesta.pop("valor_total"))

corr = amostra_cesta.corr()
fig, ax = plt.subplots(figsize=(max(8, len(corr) * 0.5), max(6, len(corr) * 0.45)))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=8)
fig.colorbar(im, ax=ax, shrink=.8); ax.set_title("Correlação entre features da cesta"); plt.tight_layout(); plt.show()

pares = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
         .rename("corr").reset_index().assign(abs_=lambda d: d["corr"].abs())
         .sort_values("abs_", ascending=False).drop(columns="abs_"))
print("Pares mais correlacionados:"); display(pares.head(10))

# Prova da colinearidade: soma das pct_* por cesta
soma = amostra_cesta[COLS_PCT].sum(axis=1)
print(f"Soma das pct_* por cesta: mín {soma.min():.4f} | máx {soma.max():.4f} (≈ 1 em todas → colinear)")

In [0]:
# Célula 8c — Estabilidade do mix ao longo do ano (participação média de cada categoria por mês do ano)
mix_mes = (cesta.groupBy("mes").agg(*[F.avg(c).alias(c) for c in COLS_PCT]).orderBy("mes").toPandas().set_index("mes"))
variacao = ((mix_mes.max() - mix_mes.min()) / mix_mes.mean()).sort_values(ascending=False)
print("Variação relativa (máx − mín) / média da participação ao longo dos meses — valores altos = categoria sazonal:")
display(variacao.rename("variação relativa").to_frame().head(10))

fig, ax = plt.subplots(figsize=(14, 5))
mix_mes[variacao.index[:6]].plot(ax=ax, marker="o")
ax.set_title("Participação média das 6 categorias mais sazonais, por mês"); ax.grid(alpha=.3); plt.show()

## 9. Checklist de adequação e conclusões
A célula abaixo resume, de forma automática, se a Silver está pronta para a etapa de ML. Os limites
(por exemplo, até 1% de itens sem categoria) são **propostos** e devem ser validados com o professor.

In [0]:
# Célula 9 — Checklist automático
pct_sem_cat = float(sem.itens.sum() / total_itens * 100) if len(sem) else 0.0
pct_flag_valor = float(tab_flags.loc[0, "% receita"])
raras = int(tab_esp["rara (<5% das cestas)"].sum())

checks = [
    ("Chave id_item_venda única", chaves == total_itens, f"{chaves:,} distintas"),
    ("Sem nulos em colunas obrigatórias",
     int(tab_nulos.drop(index=["unidade_medida"], errors="ignore").nulos.sum()) == 0, "exceto unidade_medida"),
    ("Sem meses faltando na série", len(faltantes) == 0, f"{len(faltantes)} faltando"),
    ("Itens sem categoria_pai ≤ 1%", pct_sem_cat <= 1, f"{pct_sem_cat:.2f}%"),
    ("SKUs sem conflito de categoria", conflitos == 0, f"{conflitos:,} conflitos"),
    ("flag_valor_inconsistente ≤ 1% da receita", pct_flag_valor <= 1, f"{pct_flag_valor:.2f}%"),
    ("Categorias raras (<5% das cestas)", raras == 0, f"{raras} — avaliar agrupar em 'outras'"),
    ("log1p reduz assimetria do valor_total",
     abs(perfil["skew_log1p_valor_total"]) < abs(perfil["skew_valor_total"]),
     f"{perfil['skew_valor_total']:.2f} → {perfil['skew_log1p_valor_total']:.2f}"),
]
display(pd.DataFrame(checks, columns=["verificação", "ok", "detalhe"])
        .assign(ok=lambda d: d.ok.map({True: "✅", False: "⚠️"})))

### Conclusões (preencher após a execução)
**Adequação dos dados**
- _…_

**Conhecendo a tabela**
- _…_

**Insights para o modelo (P2 · GMM)**
- _…_

### Decisões em aberto para a etapa de ML (alinhar com a dupla 2 e o professor)
| Decisão | Opções |
|---|---|
| Cestas com flag de valor ou fracionada | manter / excluir |
| Itens sem `categoria_pai` | excluir / categoria `sem_categoria` |
| Categorias raras | manter / agrupar em `outras` |
| Colinearidade das `pct_*` | retirar uma categoria / CLR / PCA |
| Escala de `n_itens` e `valor_total` | `log1p` + padronização |
| Cestas de 1 item | manter / tratar à parte |
| Tamanho da base para o GMM | base completa no Spark / amostra estratificada por mês no scikit-learn |